# تست تشخیصی: بررسی مقادیر واقعی بردارهای جابجایی fractional روی چند ماده

## چرا این تست؟
در Notebook 18 (نسخه‌ی اصلاح‌شده)، با وجود تصحیح shape mismatch، مدل از همان epoch ۱۰ روی
`MSE≈0.855` ثابت ماند (بدون کاهش بیشتر تا epoch ۵۰۰) — یک نشانه‌ی کلاسیک "مدل خاموش/گیر کرده"،
نه underfitting معمولی.

فرضیه: بردارهای جابجایی fractional ممکن است سیگنال کافی برای تفکیک ۹ تصویر از هم نداشته
باشند (مثلاً مقادیر بسیار کوچک، یا فقط در یک محور تغییر کنند، یا نزدیک صفر باشند که با
`SiLU` در انتهای `displacement_encoder` به مقادیر یکسان نگاشت شوند).

این نوت‌بوک، بدون هیچ آموزش مدلی، فقط بردارهای جابجایی واقعی چند ماده‌ی مختلف را استخراج و
چاپ می‌کند تا ببینیم آیا واقعاً سیگنال معناداری برای تفکیک ۹ تصویر وجود دارد یا نه.

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [ ]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
print(f"تعداد مواد یافت‌شده: {len(common)}")

# چند ماده‌ی متفاوت برای تست (نه فقط یکی)
test_formulas = common[:5] + common[len(common)//2:len(common)//2+3] + common[-3:]
test_formulas = list(dict.fromkeys(test_formulas))  # حذف تکراری
print(f"مواد تستی ({len(test_formulas)}): {test_formulas}")

## تابع استخراج بردار جابجایی (دقیقاً مثل Notebook 18)

In [ ]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


def load_material_with_phonopy(yaml_path, fc_path):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    n1 = len(symbols)

    fc = parse_FORCE_CONSTANTS(str(fc_path))
    n2 = fc.shape[1]
    if fc.shape[0] != n1:
        return None

    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

    segment_len = data['segment_nqpoint'][0]
    q_idx = min(segment_len // 2, len(data['phonon']) - 1)
    q_test = data['phonon'][q_idx]['q-position']
    real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

    dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2)
    if dim is None or err > 0.01:
        return None

    phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon.force_constants = fc
    return {'phonon': phonon, 'supercell_dim': dim, 'dim_match_error': err}


def extract_image_displacement_vectors(phonon, n_atoms, n_images):
    supercell = phonon.supercell
    sc_cart = supercell.positions
    sc_lattice = supercell.cell
    s2u_map = supercell.s2u_map

    unique_reps = np.unique(s2u_map)
    if len(unique_reps) != n_atoms:
        return None, "تعداد گروه نامنطبق"

    groups = {rep: np.where(s2u_map == rep)[0] for rep in unique_reps}
    if not all(len(v) == n_images for v in groups.values()):
        return None, "اندازه‌ی گروه نامنطبق"

    sorted_displacements_per_group = []
    for rep in unique_reps:
        members = groups[rep]
        disp = sc_cart[members] - sc_cart[rep]
        norms = np.linalg.norm(disp, axis=1)
        order = np.argsort(norms)
        sorted_displacements_per_group.append(disp[order])
    sorted_displacements_per_group = np.array(sorted_displacements_per_group)

    std_across_groups = np.std(sorted_displacements_per_group, axis=0)
    max_std = np.max(std_across_groups)
    if max_std > 1e-2:
        return None, f"ناسازگاری بین گروه‌ها (max_std={max_std:.6f})"

    mean_disp_cart = np.mean(sorted_displacements_per_group, axis=0)

    try:
        inv_lattice = np.linalg.inv(sc_lattice)
    except np.linalg.LinAlgError:
        return None, "لاتیس singular"
    mean_disp_frac = mean_disp_cart @ inv_lattice

    return mean_disp_frac.astype(np.float32), "OK"

print("✅ توابع آماده شدند")

## استخراج و چاپ بردارهای جابجایی برای هر ماده‌ی تستی

In [ ]:
N_ATOMS_FIXED = 8
N_IMAGES_FIXED = 9

for formula in test_formulas:
    print(f"\n{'='*60}")
    print(f"ماده: {formula}")
    try:
        result = load_material_with_phonopy(band_yaml_files[formula], fc_files[formula])
        if result is None:
            print("  ❌ load_material_with_phonopy شکست خورد")
            continue

        print(f"  supercell_dim کشف‌شده: {result['supercell_dim']} (خطا: {result['dim_match_error']:.6f})")

        disp_vectors, status = extract_image_displacement_vectors(
            result['phonon'], N_ATOMS_FIXED, N_IMAGES_FIXED)

        if disp_vectors is None:
            print(f"  ❌ استخراج بردار جابجایی شکست خورد: {status}")
            continue

        print(f"  ✅ بردارهای جابجایی fractional (shape={disp_vectors.shape}):")
        for k, v in enumerate(disp_vectors):
            print(f"     image {k}: [{v[0]:+.4f}, {v[1]:+.4f}, {v[2]:+.4f}]  (norm={np.linalg.norm(v):.4f})")

        # آماره‌های کلی برای ارزیابی کیفیت سیگنال
        all_norms = np.linalg.norm(disp_vectors, axis=1)
        print(f"  📊 norm: min={all_norms.min():.4f}, max={all_norms.max():.4f}, std={all_norms.std():.4f}")

        # آیا بردارها از هم متمایزند؟ (فاصله‌ی زوجی حداقل)
        from scipy.spatial.distance import pdist
        pairwise_dists = pdist(disp_vectors)
        print(f"  📊 حداقل فاصله‌ی زوجی بین بردارها: {pairwise_dists.min():.6f} "
              f"(اگر نزدیک صفر باشد، بردارها قابل تفکیک نیستند)")

    except Exception as e:
        print(f"  ❌ خطای غیرمنتظره: {e}")

## نتیجه‌گیری این تست

به موارد زیر در خروجی بالا دقت کنید:
1. **آیا بردارهای جابجایی بین مواد مختلف یکسان هستند؟** (مثلاً همیشه `[0,0,0], [0,0,0.111], ...`)
   اگر بله، یعنی این بردار فقط نوعی "شماره‌گذاری ترتیبی" است (شبیه ایندکس Notebook 17)، نه
   سیگنال فیزیکی متفاوت برای مواد مختلف — که می‌تواند توضیح دهد چرا کمک نکرد!
2. **آیا حداقل فاصله‌ی زوجی بین ۹ بردار، عدد معناداری است (نه خیلی نزدیک صفر)؟**
   اگر بردارها به‌خوبی از هم متمایزند، مشکل جای دیگری است (مثلاً نرمال‌سازی ورودی شبکه یا
   نرخ یادگیری).
3. **آیا `supercell_dim` بین مواد مختلف متفاوت است؟** (مثلاً برخی `(1,1,9)` و برخی `(3,3,1)`)
   اگر بله، باید توجه کنیم که "image k" در یک ماده با `(1,1,9)` به‌کلی معنای متفاوتی نسبت
   به ماده‌ای با `(3,3,1)` دارد — این می‌تواند یک منبع جدی ناهماهنگی باشد، چون مدل با یک
   `displacement_encoder` مشترک بین همه‌ی مواد کار می‌کند ولی این بردارها مقیاس و معنای
   متفاوتی بین مواد مختلف دارند (به‌خصوص اگر فاصله‌ی فیزیکی واقعی Å بین تصاویر، در مواد
   مختلف خیلی متفاوت باشد، در حالی که در fractional طبیعتاً بین ۰ و ۱ نرمال شده‌اند بدون
   در نظر گرفتن اندازه‌ی واقعی لاتیس).

**لطفاً کل خروجی این تست را برای من بفرستید.**